In [2]:
# ============================================================
# IMPORTAÇÕES
# ============================================================

import time
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    precision_recall_curve,
    auc,
    log_loss
)


# ============================================================
# FUNÇÃO PARA TRUNCAR EM 6 CASAS DECIMAIS
# ============================================================

def truncar_6(x):
    """
    Trunca um número em 6 casas decimais, sem arredondar.
    """
    return np.trunc(float(x) * 1_000_000) / 1_000_000


# ============================================================
# LEITURA DO CSV
# ============================================================

df = pd.read_csv("creditcard.csv")


# ============================================================
# DEFINIÇÃO DO TARGET
# ============================================================

if "status_fraude" in df.columns:
    target_name = "status_fraude"

elif "Class" in df.columns:
    df = df.rename(columns={"Class": "status_fraude"})
    target_name = "status_fraude"

else:
    raise ValueError("Não encontrei a coluna target: 'status_fraude' ou 'Class'.")


# ============================================================
# SELEÇÃO DAS FEATURES NUMÉRICAS
# ============================================================

features = [
    col for col in df.columns
    if col != target_name
    and pd.api.types.is_numeric_dtype(df[col])
]


print("Dataset carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")
print(f"Target utilizado: {target_name}")
print(f"Quantidade de features numéricas analisadas: {len(features)}")


# ============================================================
# LOG-PDF GAUSSIANA MULTIVARIADA
# ============================================================

def logpdf_gaussiana_multivariada(X, media, cov):
    """
    Calcula log N(x | media, cov).

    Funciona para 1D e também para múltiplas dimensões.
    """

    X = np.asarray(X)
    media = np.asarray(media)
    cov = np.asarray(cov)

    if X.ndim == 1:
        X = X.reshape(-1, 1)

    if media.ndim == 0:
        media = np.array([media])

    media = media.reshape(-1)

    cov = np.atleast_2d(cov)

    n_features = X.shape[1]

    sinal, logdet = np.linalg.slogdet(cov)

    if sinal <= 0:
        return np.full(X.shape[0], -np.inf)

    diff = X - media

    solucao = np.linalg.solve(cov, diff.T).T

    termo_quadratico = np.sum(diff * solucao, axis=1)

    logpdf = -0.5 * (
        n_features * np.log(2 * np.pi)
        + logdet
        + termo_quadratico
    )

    return logpdf


# ============================================================
# LOG-VEROSSIMILHANÇA COM RÓTULO
# ============================================================

def calcular_log_veross_com_rotulo(
    X_scaled,
    y_real,
    reg_covar=1e-6
):
    """
    Calcula a log-verossimilhança média por amostra usando o rótulo real.

    Ideia:
    - y = 0 define uma Gaussiana para não fraude
    - y = 1 define uma Gaussiana para fraude
    - os pesos são as proporções reais das classes

    Retorna:
    - log-verossimilhança média por amostra.
    """

    X_scaled = np.asarray(X_scaled)

    if X_scaled.ndim == 1:
        X_scaled = X_scaled.reshape(-1, 1)

    y_real = np.asarray(y_real).astype(int)

    n_amostras, n_features = X_scaled.shape

    log_veross_total = 0.0

    for classe in [0, 1]:

        X_classe = X_scaled[y_real == classe]

        n_classe = X_classe.shape[0]

        if n_classe <= 1:
            return np.nan

        peso_classe = n_classe / n_amostras

        media_classe = np.mean(
            X_classe,
            axis=0
        )

        cov_classe = np.cov(
            X_classe,
            rowvar=False
        )

        cov_classe = np.atleast_2d(cov_classe)

        cov_classe = cov_classe + reg_covar * np.eye(n_features)

        logpdf_classe = logpdf_gaussiana_multivariada(
            X=X_classe,
            media=media_classe,
            cov=cov_classe
        )

        log_veross_total += np.sum(
            np.log(peso_classe) + logpdf_classe
        )

    log_veross_media = log_veross_total / n_amostras

    return log_veross_media


# ============================================================
# FUNÇÃO OTIMIZADA PARA ENCONTRAR O MELHOR PONTO DE CORTE
# PELO MCC USANDO AS PRÓPRIAS PROBABILIDADES COMO THRESHOLDS
# ============================================================

def encontrar_melhor_ponto_corte_mcc(y_real, probabilidades):
    """
    Encontra o melhor ponto de corte pelo MCC.

    Usa como thresholds as próprias probabilidades estimadas pelo modelo,
    mas calcula tudo de forma otimizada via ordenação e somas acumuladas.

    Regra:
        y_pred = 1 se probabilidade >= threshold
        y_pred = 0 caso contrário
    """

    y_real = np.asarray(y_real).astype(int)
    probabilidades = np.asarray(probabilidades)

    ordem = np.argsort(-probabilidades)

    probs_ord = probabilidades[ordem]
    y_ord = y_real[ordem]

    total_positivos = np.sum(y_ord == 1)
    total_negativos = np.sum(y_ord == 0)

    tp_acum = np.cumsum(y_ord == 1)
    fp_acum = np.cumsum(y_ord == 0)

    fn_acum = total_positivos - tp_acum
    tn_acum = total_negativos - fp_acum

    numerador = (tp_acum * tn_acum) - (fp_acum * fn_acum)

    denominador = np.sqrt(
        (tp_acum + fp_acum) *
        (tp_acum + fn_acum) *
        (tn_acum + fp_acum) *
        (tn_acum + fn_acum)
    )

    mccs = np.divide(
        numerador,
        denominador,
        out=np.zeros_like(numerador, dtype=float),
        where=denominador != 0
    )

    indices_validos = np.r_[
        np.where(probs_ord[:-1] != probs_ord[1:])[0],
        len(probs_ord) - 1
    ]

    mccs_validos = mccs[indices_validos]

    melhor_idx_local = np.argmax(mccs_validos)
    melhor_idx = indices_validos[melhor_idx_local]

    melhor_ponto_corte = probs_ord[melhor_idx]
    melhor_mcc = mccs[melhor_idx]

    return melhor_ponto_corte, melhor_mcc


# ============================================================
# FUNÇÃO PRINCIPAL PARA ANALISAR UMA FEATURE 1x1
# ============================================================

def analisar_feature_flexivel(
    df,
    feature,
    target_name="status_fraude",
    resumo_features=None,
    verbose=False
):

    if resumo_features is None:
        resumo_features = []

    inicio = time.perf_counter()

    if verbose:
        print(f"\nIniciando feature: {feature}")

    # ========================================================
    # DADOS
    # ========================================================

    temp = df[[feature, target_name]].dropna()

    if temp.empty:
        if verbose:
            print("  - Ignorada: dados vazios após dropna")
        return resumo_features

    X = temp[[feature]]
    y_real = temp[target_name].astype(int)

    if y_real.nunique() < 2:
        if verbose:
            print("  - Ignorada: target possui apenas uma classe")
        return resumo_features

    if X[feature].nunique() < 2:
        if verbose:
            print("  - Ignorada: feature constante")
        return resumo_features

    # ========================================================
    # ESCALONAMENTO
    # ========================================================

    if verbose:
        print("  - Escalonando feature")

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # ========================================================
    # TREINAMENTO GMM
    # ========================================================

    if verbose:
        print("  - Treinando GMM")

    try:
        gmm = GaussianMixture(
            n_components=2,
            covariance_type="full",
            random_state=42,
            n_init=3,
            reg_covar=1e-6
        )

        gmm.fit(X_scaled)

    except Exception as erro:
        if verbose:
            print(f"  - Erro no treinamento da GMM: {erro}")
        return resumo_features

    # ========================================================
    # LOG-VEROSSIMILHANÇA DA GMM SEM RÓTULO
    # ========================================================
    # gmm.score(X_scaled) retorna a log-verossimilhança média
    # por amostra segundo a GMM ajustada sem usar o rótulo.

    log_veross_gmm = gmm.score(X_scaled)

    # ========================================================
    # LOG-VEROSSIMILHANÇA COM RÓTULO
    # ========================================================
    # Aqui o rótulo real define as duas Gaussianas:
    # uma para y=0 e outra para y=1.

    log_veross_com_rotulo = calcular_log_veross_com_rotulo(
        X_scaled=X_scaled,
        y_real=y_real,
        reg_covar=1e-6
    )

    # ========================================================
    # NEGATIVE LOG-LIKELIHOODS
    # ========================================================
    # Agora trabalhamos com -log-verossimilhança.
    # Nesse caso, menor é melhor.

    neg_log_veross_com_rotulo = -log_veross_com_rotulo
    neg_log_veross_gmm = -log_veross_gmm

    diferenca_neg_log_veross = (
        neg_log_veross_com_rotulo
        - neg_log_veross_gmm
    )

    # ========================================================
    # CLUSTERS
    # ========================================================

    if verbose:
        print("  - Identificando cluster associado à fraude")

    clusters = gmm.predict(X_scaled)

    ct = pd.crosstab(
        clusters,
        y_real
    )

    if 1 not in ct.columns:
        if verbose:
            print("  - Ignorada: classe 1 não encontrada no crosstab")
        return resumo_features

    cluster_fraude = ct[1].idxmax()

    # ========================================================
    # PROBABILIDADES DO CLUSTER ASSOCIADO À FRAUDE
    # ========================================================

    if verbose:
        print("  - Calculando probabilidades")

    probabilidades = gmm.predict_proba(X_scaled)[:, cluster_fraude]

    probabilidades = np.clip(
        probabilidades,
        1e-15,
        1 - 1e-15
    )

    # ========================================================
    # AUC-PR
    # ========================================================

    if verbose:
        print("  - Calculando AUC-PR")

    precision_vals, recall_vals, _ = precision_recall_curve(
        y_real,
        probabilidades
    )

    auc_pr = auc(
        recall_vals,
        precision_vals
    )

    # ========================================================
    # MELHOR PONTO DE CORTE E MCC OTIMIZADO
    # ========================================================

    if verbose:
        print("  - Otimizando ponto de corte pelo MCC")

    melhor_ponto_corte, mcc = encontrar_melhor_ponto_corte_mcc(
        y_real=y_real,
        probabilidades=probabilidades
    )

    # ========================================================
    # PONTO DE CORTE MÉDIO FIXO
    # ========================================================

    ponto_corte_medio = 0.5

    # ========================================================
    # LOG LOSS
    # ========================================================

    if verbose:
        print("  - Calculando Log Loss")

    ll = log_loss(
        y_real,
        probabilidades
    )

    fim = time.perf_counter()

    # ========================================================
    # NORMALIZAÇÕES ENTRE 0 E 1
    # ========================================================

    auc_pr_norm = np.clip(
        auc_pr,
        0,
        1
    )

    mcc_norm = (mcc + 1) / 2

    mcc_norm = np.clip(
        mcc_norm,
        0,
        1
    )

    log_loss_norm = 1 / (1 + ll)

    log_loss_norm = np.clip(
        log_loss_norm,
        0,
        1
    )

    # ========================================================
    # SCORE FINAL
    # ========================================================

    score_final = np.mean([
        auc_pr_norm,
        mcc_norm,
        log_loss_norm
    ])

    # ========================================================
    # RESULTADO COM TRUNCAMENTO EM 6 CASAS
    # ========================================================

    nova_linha = {
        "Feature": feature,
        "AUC_PR": truncar_6(float(auc_pr)),
        "MCC": truncar_6(float(mcc)),
        "Log_Loss": truncar_6(float(ll)),
        "Log_Loss_Norm": truncar_6(float(log_loss_norm)),

        "Neg_Log_Veross_Com_Rotulo": truncar_6(float(neg_log_veross_com_rotulo)),
        "Neg_Log_Veross_GMM": truncar_6(float(neg_log_veross_gmm)),
        "Diferenca_Neg_Log_Veross": truncar_6(float(diferenca_neg_log_veross)),

        "Score_Final": truncar_6(float(score_final)),
        "Melhor_Ponto_Corte": truncar_6(float(melhor_ponto_corte)),
        "Ponto_Corte_Medio": truncar_6(float(ponto_corte_medio)),
        "Tempo": truncar_6(float(fim - inicio))
    }

    resumo_features.append(nova_linha)

    if verbose:
        print(f"  - Finalizada em {fim - inicio:.2f} segundos")
        print(f"  - Score_Final: {score_final:.6f}")
        print(f"  - Neg_Log_Veross_Com_Rotulo: {neg_log_veross_com_rotulo:.6f}")
        print(f"  - Neg_Log_Veross_GMM: {neg_log_veross_gmm:.6f}")
        print(f"  - Diferenca_Neg_Log_Veross: {diferenca_neg_log_veross:.6f}")

    return resumo_features


# ============================================================
# EXECUÇÃO PARA TODAS AS FEATURES COM BARRA DE PROGRESSO
# ============================================================

resumo_features = []

inicio_geral = time.perf_counter()

for feature in tqdm(
    features,
    desc="Processando features 1x1",
    unit="feature"
):
    tamanho_antes = len(resumo_features)

    resumo_features = analisar_feature_flexivel(
        df=df,
        feature=feature,
        target_name=target_name,
        resumo_features=resumo_features,
        verbose=False
    )

    tamanho_depois = len(resumo_features)

    if tamanho_depois > tamanho_antes:
        tqdm.write(f"Feature processada: {feature}")
    else:
        tqdm.write(f"Feature ignorada ou com erro: {feature}")

fim_geral = time.perf_counter()


# ============================================================
# DATAFRAME FINAL
# ============================================================

scores_1x1 = pd.DataFrame(resumo_features)

if scores_1x1.empty:
    raise ValueError(
        "Nenhuma feature foi processada. "
        "Verifique se existem features numéricas válidas e se o target está correto."
    )

scores_1x1 = scores_1x1.sort_values(
    by="Score_Final",
    ascending=False
).reset_index(drop=True)

scores_1x1["Posicao_Rank"] = np.arange(
    1,
    len(scores_1x1) + 1
)

scores_1x1 = scores_1x1[
    [
        "Feature",
        "AUC_PR",
        "MCC",
        "Log_Loss",
        "Log_Loss_Norm",
        "Neg_Log_Veross_Com_Rotulo",
        "Neg_Log_Veross_GMM",
        "Diferenca_Neg_Log_Veross",
        "Score_Final",
        "Melhor_Ponto_Corte",
        "Ponto_Corte_Medio",
        "Tempo",
        "Posicao_Rank"
    ]
]


# ============================================================
# EXPORTAÇÃO PARA CSV
# ============================================================

scores_1x1.to_csv(
    "1x1_scores.csv",
    index=False,
    encoding="utf-8-sig",
    float_format="%.6f"
)


# ============================================================
# RELATÓRIO FINAL
# ============================================================

tempo_total_segundos = fim_geral - inicio_geral
tempo_total_minutos = tempo_total_segundos / 60

print("\nProcessamento finalizado.")
print(f"Features processadas com sucesso: {len(scores_1x1)}")
print(f"Tempo total: {tempo_total_segundos:.2f} segundos")
print(f"Tempo total: {tempo_total_minutos:.2f} minutos")
print("Arquivo salvo como: 1x1_scores.csv")

print("\nTop 20 features:")
display(scores_1x1.head(20))

Dataset carregado com sucesso.
Shape do dataset: (283726, 31)
Target utilizado: status_fraude
Quantidade de features numéricas analisadas: 30


Processando features 1x1:   0%|          | 0/30 [00:00<?, ?feature/s]

Feature processada: tempo_desde_a_primeira_transacao
Feature processada: V1
Feature processada: V2
Feature processada: V3
Feature processada: V4
Feature processada: V5
Feature processada: V6
Feature processada: V7
Feature processada: V8
Feature processada: V9
Feature processada: V10
Feature processada: V11
Feature processada: V12
Feature processada: V13
Feature processada: V14
Feature processada: V15
Feature processada: V16
Feature processada: V17
Feature processada: V18
Feature processada: V19
Feature processada: V20
Feature processada: V21
Feature processada: V22
Feature processada: V23
Feature processada: V24
Feature processada: V25
Feature processada: V26
Feature processada: V27
Feature processada: V28
Feature processada: valor_de_transacao

Processamento finalizado.
Features processadas com sucesso: 30
Tempo total: 256.27 segundos
Tempo total: 4.27 minutos
Arquivo salvo como: 1x1_scores.csv

Top 20 features:


,Feature,AUC_PR,MCC,Log_Loss,Log_Loss_Norm,Neg_Log_Veross_Com_Rotulo,Neg_Log_Veross_GMM,Diferenca_Neg_Log_Veross,Score_Final,Melhor_Ponto_Corte,Ponto_Corte_Medio,Tempo,Posicao_Rank
0,V17,0.512280,0.558214,0.154764,0.865977,1.316690,1.252407,0.064283,0.719121,0.999999,0.5,13.945954,1
1,V12,0.600688,0.686326,0.645433,0.607742,1.383071,1.290194,0.092876,0.683864,0.999999,0.5,5.080260,2
2,V14,0.572961,0.591352,0.632623,0.612511,1.371204,1.311434,0.059769,0.660382,0.999999,0.5,6.700491,3
3,V16,0.451239,0.566710,0.587378,0.629969,1.399927,1.379160,0.020766,0.621521,0.999999,0.5,11.864794,4
4,V11,0.478168,0.589107,1.422576,0.412783,1.416609,1.416020,0.000589,0.561835,0.999996,0.5,5.451677,5
5,V10,0.203260,0.239289,0.251756,0.798877,1.395836,1.229562,0.166274,0.540594,0.998563,0.5,13.538837,6
6,V18,0.318570,0.487547,0.816480,0.550515,1.418413,1.421009,-0.002595,0.537619,0.997557,0.5,7.405044,7
7,V7,0.168677,0.181559,0.273548,0.785207,1.392467,1.142027,0.250439,0.514888,0.999999,0.5,12.834656,8
8,V4,0.211976,0.359690,0.757665,0.568936,1.421366,1.382606,0.038760,0.486919,0.999994,0.5,10.791919,9
9,V3,0.219514,0.272399,0.813674,0.551366,1.399410,1.374703,0.024706,0.469026,0.999990,0.5,10.865076,10
